# Results and Discussion: OpenBind ligand-affinity modelling

This notebook presents the dissertation-ready Results and Discussion for the five-fold random and scaffold cross-validation experiments. It loads the figures and tables generated by `openbind_results_analysis.ipynb`; it does not recalculate model results. Numerical conclusions refer to pooled out-of-fold **compound-level** predictions unless fold-level performance is stated explicitly.

## Code used and shared imports

I use this notebook to present the tables and figures produced by `openbind_results_analysis.ipynb`. I run that notebook first, then work through these display cells in order.

All imports are kept here: `Path` locates the analysis folder, `pandas` reads and formats the CSV tables, and `IPython.display` embeds the saved figures and styled tables. These cells do not retrain models or recalculate metrics.


In [ ]:
# Resolve the exported analysis folder from the notebook's working directory.
from pathlib import Path

# Read the CSV summaries and format tables for notebook display.
import pandas as pd
# Embed the saved PNG figures and pandas tables in the notebook.
from IPython.display import Image, display

# Locate the MGT repository whether the notebook is opened from the project root,
# the MGT directory or the notebooks directory.
candidates = [Path.cwd(), Path.cwd() / 'MGT', Path.cwd().parent]
candidates.extend(parent for parent in Path.cwd().parents if parent.name == 'MGT')
ROOT = next((path.resolve() for path in candidates if (path / 'output' / 'dissertation_analysis').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Could not locate MGT/output/dissertation_analysis.')

ANALYSIS_ROOT = ROOT / 'output' / 'dissertation_analysis'
FIGURE_ROOT = ANALYSIS_ROOT / 'figures'
TABLE_ROOT = ANALYSIS_ROOT / 'tables'

# Fail clearly if an analysis export is missing rather than displaying a stale substitute.
def show_figure(filename: str, width: int = 1100) -> None:
    path = FIGURE_ROOT / filename
    if not path.is_file():
        raise FileNotFoundError(path)
    display(Image(filename=str(path), width=width))

# Read the upstream CSV; formatting is applied only when the table is displayed.
def read_table(filename: str) -> pd.DataFrame:
    path = TABLE_ROOT / filename
    if not path.is_file():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

print(f'Using analysis outputs from: {ANALYSIS_ROOT}')

# 3 Results

The experiments tested whether progressively richer molecular representations improved pKD prediction for a single protease target. Seven configurations formed a controlled hierarchy: Morgan MLP, 2D GNN, crystallographic 3D distance GNN, 3D ALIGNN, adapted MGT, masked ALIGNN and masked MGT. Every compound contributed to one held-out outer fold, and repeated crystal structures were combined into one compound-level prediction.

## 3.1 Data

### 3.1.1 Data collection and curation

The OpenBind release contained 925 crystallographic binding events from 699 compounds, of which 601 were reported to have affinity measurements [1]. Linking the structures to the official filtered affinity reference, then excluding missing labels, covalent ligands, suspected crystallographic artefacts, failed reference poses and identity mismatches, produced 621 structures representing 474 compounds. The 304 excluded structure records do not equal the sum of individual reason counts because a record could fail more than one rule. Structure-file auditing found all expected files before quality filtering, so attrition arose from the predefined modelling criteria rather than missing downloads.

Most compounds had one retained structure (345/474), while 129 had two to four structures. Grouping these repeats throughout splitting prevented the same compound entering more than one partition; averaging their held-out predictions also prevented compounds with several crystal poses from receiving disproportionate weight.

### Code used: Curation tables

I load the saved curation and exclusion tables and format their counts as whole numbers for display. The underlying CSV values are unchanged.


In [ ]:
# Format counts for the chapter without modifying the saved curation tables.
curation = read_table('table_dataset_curation.csv')
exclusions = read_table('table_exclusion_reasons.csv')
display(curation.style.format(precision=0))
display(exclusions.style.format({'Record count': '{:,.0f}'}))

**Table 3.1. Dataset curation.** Progression from the OpenBind release to the frozen modelling cohort and the overlapping reasons for excluding structure records.

### 3.1.2 Affinity and chemical-space distribution

The 474 compound labels spanned pKD 3.437–7.938, with mean 5.118, median 4.942 and population SD 0.876 (Figure 3.1). Thus, the study covered approximately 4.5 log units of affinity, although observations were concentrated near pKD 4.5–5.2. Each outer test fold retained approximately one fifth of the compounds.

The fingerprint projection showed broad overlap between random-fold colours, as expected after pKD-stratified compound assignment (Figure 3.2). Mean pairwise Morgan Tanimoto similarity was almost identical within and between random folds (0.3093 and 0.3095). Under scaffold CV it was modestly higher within folds than between folds (0.3225 versus 0.3062). This difference is small because Morgan similarity and exact Bemis–Murcko scaffold identity are not equivalent; nevertheless, the scaffold manifests enforced zero scaffold overlap, making them the stricter test of transfer to unseen cores.

### Code used: Affinity distribution

This cell displays the compound-level pKD distribution exported by the analysis notebook; it does not rebin the data.


In [ ]:
# Use the exported compound-level distribution, not a new structure-level histogram.
show_figure('figure_pkd_distribution.png', width=1050)

**Figure 3.1. Compound-level pKD distribution.** Distribution of the frozen affinity endpoint across the complete cohort and the five outer test folds.

### Code used: Chemical-space panel

I load the existing fingerprint projection and fold-similarity figure so the displayed panel uses the same compound set as the analysis.


In [ ]:
# Read the fingerprint/fold panel generated from the frozen analysis cohort.
show_figure('figure_chemical_space_folds.png', width=1200)

**Figure 3.2. Chemical-space allocation.** Morgan-fingerprint projection and within-/between-fold Tanimoto distributions for random and scaffold five-fold CV. The first two projection components explain 18.9% of fingerprint variance and are illustrative rather than a complete map of chemical space.

## 3.2 Molecular affinity models

### Code used: Random and scaffold performance tables

I select the same metric columns from both CV summaries and round only the displayed values. Fold variability and pooled compound metrics remain separate columns.


In [ ]:
# Keep fold mean/SD distinct from metrics pooled across out-of-fold compounds.
columns = ['Model', 'Fold RMSE mean', 'Fold RMSE SD', 'Pooled MAE', 'Pooled RMSE', 'Pooled R2', 'Pooled Spearman']
random_performance = read_table('table_random_cv_performance.csv')[columns]
scaffold_performance = read_table('table_scaffold_cv_performance.csv')[columns]
display(random_performance.style.format(precision=3).set_caption('Table 3.2a. Random five-fold CV'))
display(scaffold_performance.style.format(precision=3).set_caption('Table 3.2b. Scaffold five-fold CV'))

### 3.2.1 Random five-fold cross-validation

Random CV produced a clear gain when crystallographic distances were introduced (Table 3.2a; Figure 3.3). Pooled RMSE fell from 0.644 for the Morgan MLP and 0.620 for the 2D GNN to 0.570 for the 3D distance GNN. ALIGNN gave the lowest pooled RMSE, 0.568, and the highest R², 0.580, but improved on the distance GNN by only 0.0025 pKD. Its paired bootstrap interval relative to the distance GNN crossed zero (−0.0270 to 0.0223), so these two models formed the same statistical performance group. By contrast, the Morgan MLP, 2D GNN and adapted MGT were worse than the distance GNN with intervals excluding zero.

Fold results reinforce this interpretation. The 3D distance GNN was best in random folds 1, 2 and 4, while ALIGNN was best in folds 0 and 3. Therefore, no MGT configuration won a random fold. The pooled result supports crystallographic distance information, but provides no evidence that additional angular or Coulomb-attention complexity reliably improves random-split prediction.

### Code used: Fold-level comparison

This reads the saved fold-RMSE plot rather than recomputing metrics from the prediction files.


In [ ]:
# The fold scores and error bars are already encoded in this image.
show_figure('figure_cv_model_rmse.png', width=1200)

**Figure 3.3. Model performance across folds.** Compound-level outer-fold RMSE values with fold mean ± sample SD. Pooled values in Table 3.2 are calculated from all 474 non-overlapping out-of-fold predictions and therefore need not equal the unweighted fold mean.

### 3.2.2 Scaffold five-fold cross-validation

The scaffold ranking was similar at the pooled level: the 3D distance GNN was best (RMSE 0.576; R² 0.568), followed by ALIGNN (0.583), masked ALIGNN (0.591) and adapted MGT (0.600). However, fold-level winners were more heterogeneous. The distance GNN won folds 0 and 4, the 2D GNN narrowly won fold 1, masked ALIGNN won fold 2, and adapted MGT won fold 3. The latter two results show that richer encoders could be advantageous for particular held-out scaffold collections even though they did not lead overall. Without a chemical-family-specific follow-up, this heterogeneity should not be assigned to a particular structural motif.

Adapted MGT is important for a second reason: its pooled RMSE was 0.602 in random CV and 0.600 in scaffold CV, a change of −0.0028 pKD. Among the crystallographic 3D models this was the smallest random-to-scaffold change, compared with +0.0055 for the distance GNN and +0.0149 for ALIGNN. Its fold RMSE SD was also nearly unchanged (0.0578 random; 0.0569 scaffold). MGT therefore showed stable aggregate behaviour across evaluation designs despite not achieving the best accuracy. This separates **robustness of ranking across split designs** from **minimum prediction error** and provides a more complete evaluation than one split alone.

Paired scaffold-cluster bootstrap intervals placed ALIGNN, adapted MGT and masked ALIGNN in the same uncertainty group as the distance GNN, whereas masked MGT was clearly worse. These exploratory intervals quantify uncertainty in held-out compounds/scaffolds, but not variation from retraining with different initialisation seeds.

### Code used: Paired uncertainty and split comparison

I display the paired-bootstrap and split-difference exports together. Resampling and metric differences were calculated upstream.


In [ ]:
# Load the upstream uncertainty estimates without resampling a second time.
show_figure('figure_paired_bootstrap_forest.png', width=1150)
show_figure('figure_random_scaffold_delta.png', width=950)

**Figure 3.4. Paired uncertainty analysis and split sensitivity.** Top: RMSE differences relative to the 3D distance GNN from 5,000 paired bootstrap resamples; random CV resampled compounds and scaffold CV resampled complete scaffold clusters. Bottom: change in pooled RMSE from random to scaffold CV.

### 3.2.3 Effect of masked pretraining

Masked atom-feature reconstruction did not improve generalisation consistently (Figure 3.5). For ALIGNN, masking increased RMSE by 0.0176 in random CV and 0.0078 in scaffold CV. For MGT, it improved random RMSE by 0.0138 but worsened scaffold RMSE by 0.0178. Masked ALIGNN nevertheless had similar, relatively low fold SD under random and scaffold CV (0.0522 and 0.0513), suggesting stable but not more accurate predictions. The direction of the MGT effect reversed between evaluation designs, so a general pretraining benefit cannot be claimed.

### Code used: Masking comparison

This cell loads the saved masked-versus-unmasked comparison; it does not perform pretraining or select a checkpoint.


In [ ]:
# Reuse the saved comparison rather than recalculating masked-model differences.
show_figure('figure_masking_effect.png', width=900)

**Figure 3.5. Effect of masked atom-feature pretraining.** Negative values favour masking; positive values indicate increased pooled RMSE after pretraining.

### 3.2.4 Learning behaviour and prediction errors

Training loss generally continued to fall after validation loss had flattened, particularly for the higher-capacity ALIGNN and MGT models (Figures 3.6–3.7). Selected MGT epochs were relatively early: 18–21 in random CV and 14–33 in scaffold CV, compared with 28–46 and 36–103 for the distance GNN. Early stopping therefore prevented later training improvements being mistaken for generalisation.

Across all models, predicted-versus-experimental plots showed compressed prediction ranges: weak ligands tended to be overpredicted and strong ligands underpredicted (Figures 3.8–3.9). This regression towards the training mean is consistent with a small dataset and limited examples at the affinity extremes. The fold-specific ligand panels show that the largest errors occurred in several folds and model winners, rather than being attributable to one failed partition (Figure 3.10).

### Code used: Learning curves

I display separate random and scaffold learning-curve panels so their validation traces are not mixed.


In [ ]:
# Keep each CV design's validation history in its own panel.
show_figure('figure_learning_curves_random_2x2.png', width=1250)
show_figure('figure_learning_curves_scaffold_2x2.png', width=1250)

**Figures 3.6–3.7. Learning behaviour.** Training and validation Huber loss for the principal crystallographic models under random and scaffold CV. Vertical markers identify the minimum-validation-loss checkpoint selected independently in each fold.

### Code used: Prediction and ligand-error panels

I load the all-model prediction plots and the saved per-fold ligand examples. Model selection and error ranking have already occurred in the analysis notebook.


In [ ]:
# Display saved prediction diagnostics and descriptive ligand-error examples.
show_figure('figure_predicted_vs_experimental_all_models_random.png', width=1250)
show_figure('figure_predicted_vs_experimental_all_models_scaffold.png', width=1250)
show_figure('figure_random_best_model_per_fold_lowest_highest_error_5x2.png', width=1050)
show_figure('figure_scaffold_best_model_per_fold_lowest_highest_error_5x2.png', width=1050)

**Figures 3.8–3.10. Out-of-fold prediction behaviour.** Pooled predicted-versus-experimental pKD for all models under both CV designs, followed by the lowest- and highest-error compounds from the best-performing model in each outer fold.

# 4 Discussion

## 4.1 Effect of molecular representation complexity

The most reproducible improvement came from adding crystallographic distances, not from continually increasing architecture complexity. Relative to the 2D GNN, the distance GNN reduced pooled RMSE by 0.0495 in random CV and 0.0459 in scaffold CV; both paired intervals excluded zero. It achieved this with 4.10 million parameters, essentially the same size as the 2D GNN. Doubling capacity to 8.12 million parameters for ALIGNN produced only a 0.0025 random improvement and a 0.0069 scaffold deterioration, with both intervals crossing zero. Increasing capacity to 13.70 million for MGT increased error. Figure 4.1 therefore identifies the distance GNN as the strongest accuracy–complexity compromise.

This partly contrasts with the original ALIGNN study, where explicit line-graph angles improved several materials and molecular-property benchmarks [3]. The present ablation shows that such benefits are task- and data-dependent: bound ligand distances were informative, but angles did not add a reliable affinity signal beyond them in this 474-compound campaign.

### Code used: Parameter-count comparison

This displays the existing parameter-count versus RMSE figure without constructing networks or recounting weights.


In [ ]:
# Parameter counts and pooled errors were calculated in the analysis notebook.
show_figure('figure_parameters_vs_rmse.png', width=1150)

**Figure 4.1. Model complexity versus pooled error.** Trainable parameter count on a logarithmic scale against compound-level RMSE; the two lowest-error models are highlighted in each CV design.

## 4.2 Why adapted MGT did not outperform the 3D models

Adapted MGT combined local distances, angular updates, a wider Coulomb graph, multi-head attention, post-ALIGNN graph convolutions and a feed-forward residual block. Yet its pooled RMSE was 0.602 random and 0.600 scaffold, versus 0.570/0.576 for the distance GNN. Three evidence-based factors may explain this result. First, MGT had more than three times the parameters of the distance GNN but only about 303–304 training compounds per fold. The learning curves show a widening training–validation gap, consistent with excess capacity relative to the available labels. Second, several MGT views may be redundant for small isolated ligands: distances and chemistry already captured much of the recoverable signal, while the Coulomb graph added another encoding of atomic identity and separation. Third, Experiment A omitted the protein. Affinity is determined by protein–ligand interactions, whereas ligand-only crystallographic geometry supplies only an indirect imprint of the bound state. MGT's wider interactions may be more useful when the graph contains pocket residues, intermolecular contacts and electrostatics; this remains a testable hypothesis, not a conclusion from the current data.

The stability of unmasked MGT across random and scaffold CV remains informative. Its low split-to-split change and a scaffold-fold win show that the encoder was functional and occasionally competitive. The result is therefore not that MGT failed, but that its additional representation complexity did not improve the mean error enough to justify its computational cost for ligand-only, single-target learning.

## 4.3 Interpretation of masked pretraining

Masked pretraining used only each fold's supervised training structures; it did not introduce a large external molecular corpus. Moreover, zeroing atom features left graph connectivity, distances, angles and—within MGT—Coulomb values visible. Reconstruction could therefore be partly solved from information correlated with elemental identity. These limitations provide plausible explanations for the mixed downstream effects. The random-CV improvement for MGT suggests that pretraining sometimes regularised its larger encoder, but the reversed scaffold effect shows that it did not learn a representation that transferred reliably to unseen cores. A stronger test would pretrain on a much larger unlabeled structure collection, mask identity-dependent edge information consistently, and compare multiple masking ratios and seeds.

## 4.4 Generalisation and comparison with previous work

Contrary to the common expectation that scaffold separation must greatly worsen performance, most pooled RMSE changes were small. This does not make scaffold CV unnecessary. Exact cores were disjoint, but the campaign contained related follow-on chemistry and similar local substructures across scaffolds; the small Tanimoto shift supports this interpretation. Reporting both designs exposed fold-specific behaviour hidden by pooled ranking, including the adapted-MGT and masked-ALIGNN scaffold wins.

The official OpenBind benchmark evaluates zero-shot/reference affinity methods on 494 compounds [2], whereas this study trained target-specific models and evaluated 474 curated compounds. Its reported affinity RMSE values were approximately 1.09–1.63 pKD, compared contextually with 0.568–0.602 for the unmasked trained 3D models (Figure 4.2). The lower values here demonstrate the advantage of fitting the target campaign, but cannot establish direct superiority because the cohorts, access to training labels and protocols differ. This distinction is central: the project evaluates representation choices under controlled supervised CV, while the released benchmark asks how transferable existing predictors are without equivalent target-specific training.

### Code used: External-reference context

I display the saved reference-method comparison. This cell does not combine the external cohort with the project's out-of-fold predictions.


In [ ]:
# Preserve the external-reference panel as contextual, not matched-CV evaluation.
show_figure('figure_openbind_benchmark_3d_models_context.png', width=1050)

**Figure 4.2. Contextual OpenBind comparison.** Released zero-shot/reference RMSE values and target-trained five-fold CV values for the crystallographic 3D GNN, ALIGNN and adapted MGT. The comparison is not head-to-head because the cohorts and evaluation protocols are different.

## 4.5 Limitations and future work

The main limitation is sample size: 474 labels are modest for 8–14 million-parameter networks. Five folds improve coverage but do not replace independent external validation, and one training seed per fold leaves optimisation variability unmeasured. The paired bootstrap captures held-out compound or scaffold uncertainty only. Experimental pKD also contains assay noise that sets an unquantified performance ceiling. Finally, using one crystallographic ligand pose without the protein prevents direct modelling of binding-pocket complementarity and means the results cannot be assumed to transfer to other targets or predicted conformers.

Future work should repeat each fold across several optimisation seeds; evaluate protein–ligand complex graphs while preserving the same compound/scaffold assignments; test whether MGT benefits specifically from intermolecular Coulomb and angular edges; and pretrain on larger external structural collections. An additional prospective or target-held-out evaluation would determine whether any gain survives beyond this campaign. On present evidence, crystallographic distances provide the clearest benefit, the distance GNN is the preferred efficiency–accuracy model, ALIGNN is statistically comparable, and adapted MGT offers split-stable but not leading pooled performance.

## References

1. OpenBind Consortium. *OpenBind Structure–Affinity Data Release: EV-A71/CVA16 2A protease*, version 1, Zenodo (2026). https://doi.org/10.5281/zenodo.20026661
2. OpenBind Consortium. *EV-A71 2A dataset and benchmarks*. https://github.com/OpenBind-Consortium/EV-A71_2A_benchmark
3. Choudhary, K. & DeCost, B. Atomistic Line Graph Neural Network for improved materials property predictions. *npj Computational Materials* **7**, 185 (2021). https://doi.org/10.1038/s41524-021-00650-1
4. MolecularGraphTransformer. *Molecular Graph Transformer (MGT) source implementation*. https://github.com/MolecularGraphTransformer/MGT